# 16 — Trendline settings research

Reads the results of `experiments/trendline_settings_search.py`, which sweeps **324 detector
settings x 3 timeframes x 2 instruments x 9 market windows** and asks four separate questions.

Run the search first (it takes roughly two hours):

```
PYTHONPATH=. python -m experiments.trendline_settings_search
```

### The four questions, and why they are kept apart

| stage | question | what it can and cannot tell you |
|---|---|---|
| **1. Quality** | which settings give well-formed, frequent lines? | **The only stable answer.** Never looks at forward returns |
| **2. Screen** | does a break predict anything? | Differenced against a shuffled null and a matched control |
| **3. Multi-timeframe** | does higher-timeframe context change a break's value? | Split by HTF slope; judge against the trade counts |
| **4. Confirmation** | does it hold on 1m and on gold? | 1m carries a 10.8%-of-R cost floor |

**The argmax of stage 2 is not a recommendation.** Re-running an identical grid after a detector fix
moved the best-scoring cell for 13 of 13 chart patterns, including ones whose ranking across all 108
settings correlated at 0.98-0.999. That surface is flat and its peak is noise, which is why settings
are chosen in stage 1 — on detection counts, never on returns — and stages 2-4 only report whether
those settings carry signal.

### Read the numbers in this order

1. **`hit20`** — the share of breaks that go the break's way. Below 0.50 and nothing else matters.
2. **`vs_null20`** — the matched edge minus the same measurement on within-day-shuffled bars. Two
   plausible control schemes have manufactured edges in this project; the shuffled null is the one
   that has reliably separated real structure from measurement artifact.
3. **`n20`** — sample size. At 30-150 breaks the standard error is large enough to swallow most of
   what you are looking at.


## 1. Load

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant

import numpy as np, pandas as pd
pd.set_option("display.width", 220)

E = "../experiments"
def load(name):
    p = f"{E}/results_trendline_{name}.csv"
    return pd.read_csv(p) if os.path.exists(p) else None

quality = load("quality"); picks = load("picks")
screen  = load("screen");  mtf   = load("mtf")
one_min = load("1m")
for nm, df in [("quality", quality), ("picks", picks), ("screen", screen),
               ("mtf", mtf), ("1m", one_min)]:
    print(f"{nm:9s} {'-' if df is None else format(len(df), ',') + ' rows'}")

## 2. Stage 1 — the settings (the stable answer)

Chosen on detection counts alone: enough breaks in **every** window, a sane break rate, and real
touch support. A line never broken faded into irrelevance; a line broken instantly was never
respected.

In [ ]:
if picks is not None:
    g = [c for c in ["atr_mult","min_touches","line_tol_atr","min_span_bars",
                     "break_buffer_atr","lookback_pivots"] if c in picks.columns]
    display(picks[["symbol","tf"] + g + ["breaks","lines","broke_frac","touches","pct_live"]]
            .round(3).sort_values(["symbol","tf"]))

### How many settings pass the quality gate at all?

A timeframe where few settings pass is telling you the instrument does not hold trendlines at that
resolution — which is a finding, not a tuning problem.

In [ ]:
if quality is not None:
    display(quality.groupby(["symbol","tf"]).quality_ok.agg(passed="sum", tried="size")
            .assign(pct=lambda d: (100*d.passed/d.tried).round(1)))

## 3. Stage 2 — does a break predict anything?

`vs_null20` is the honest column: the matched edge minus the same measurement on shuffled bars.

In [ ]:
if screen is not None:
    keep = screen[screen.n20 >= 30]
    display(keep.groupby(["symbol","tf","side"])[
        ["n20","edge20","matched20","matched20_null","vs_null20","hit20"]]
        .median().round(4))
    print(f"\ncells with n>=30: {len(keep)}")
    print(f"beating their null: {int((keep.vs_null20>0).sum())} "
          f"({100*(keep.vs_null20>0).mean():.0f}%)   chance = 50%")
    print(f"median hit rate: {keep.hit20.median():.4f}   (a directional edge needs > 0.50)")

### Which timeframe looks best, and does that survive the null?

In [ ]:
if screen is not None:
    keep = screen[screen.n20 >= 30]
    by_tf = (keep.groupby("tf")
                 .agg(cells=("vs_null20","size"), median_vs_null=("vs_null20","median"),
                      pct_positive=("vs_null20", lambda s: round(100*(s>0).mean(),1)),
                      median_hit=("hit20","median"), median_n=("n20","median"))
                 .sort_values("median_vs_null", ascending=False))
    display(by_tf.round(4))

### Which knob actually matters?

Pooled over everything, so it averages out the per-cell noise that makes a single argmax
meaningless.

In [ ]:
if screen is not None:
    keep = screen[screen.n20 >= 30]
    for k in ["atr_mult","min_touches","line_tol_atr","min_span_bars",
              "break_buffer_atr","lookback_pivots"]:
        if k in keep.columns:
            print(f"\n{k}:")
            print(keep.groupby(k).agg(cells=("vs_null20","size"),
                                      vs_null=("vs_null20","median"),
                                      hit=("hit20","median")).round(4).to_string())

## 4. Stage 3 — multi-timeframe context

The obvious test — requiring a break on both timeframes — **cannot be answered at this event rate**:
1h produces about 6 up-breaks in six months, so a 15m break coinciding with one happens 0-1 times.
That is arithmetic, not a null result.

So the higher timeframe supplies a *state* instead: is its line on the same side sloping **with** the
break or **against** it? Every lower-timeframe break gets classified, which splits hundreds of events
rather than a handful.

**Check the `n` column before believing any gap.** A split that helps only by trading far less has
not been shown to help — that is methodology rule 7, and it has retracted results here before.

In [ ]:
if mtf is not None:
    piv = (mtf.groupby(["symbol","hi_tf","lo_tf","side","kind"])
              .agg(n=("n","sum"), edge20=("edge20","median"), hit20=("hit20","median"))
              .reset_index())
    display(piv.round(4))
    w = piv.pivot_table(index=["symbol","hi_tf","lo_tf","side"], columns="kind",
                        values=["edge20","n"])
    if ("edge20","htf_aligned") in w.columns:
        w[("gain","")] = w[("edge20","htf_aligned")] - w[("edge20","htf_against")]
        display(w.round(4))

## 5. Stage 4 — 1m, where the cost floor bites hardest

Spread is **10.8% of one R** on BTC 1m at a 1.5xATR stop, which needs a **36.9% win rate at 2R** just
to break even. Every entry family measured in this project came in at 31-35%.

In [ ]:
if one_min is not None:
    display(one_min.groupby(["symbol","side"])
            .agg(n=("n20","sum"), edge20=("edge20","median"),
                 matched20=("matched20","median"), hit20=("hit20","median")).round(4))
else:
    print("no 1m results — the search skips stage 4 in --quick mode")

## 6. Draw the winning settings

Put a row from section 2 into `CFG` and look at the lines it actually produces. Every defect in this
library so far was found by looking at a chart, not by reading a number.

In [ ]:
from quant.data import get_ohlcv
from quant.viz import pattern_figure

SYMBOL, TF, SOURCE = "BTCUSDT", "15m", "binance"
START, END = "2026-07-01", "2026-08-24"

CFG = dict(atr_mult=3.5, min_touches=3, line_tol_atr=0.6,
           min_span_bars=40, break_buffer_atr=0.0, lookback_pivots=16)
if picks is not None:
    row = picks[(picks.symbol == SYMBOL) & (picks.tf == TF)]
    if len(row):
        CFG = {k: (int(row.iloc[0][k]) if k in ("min_touches","min_span_bars","lookback_pivots")
                   else float(row.iloc[0][k]))
               for k in CFG if k in row.columns}
print("drawing with:", CFG)

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
fig = pattern_figure(df, patterns=[], sr_zones=False, trendlines=True, trendline_kwargs=CFG,
                     start_visible=("support trendline", "resistance trendline"),
                     title=f"{SYMBOL} {TF} — searched settings", height=800)
fig

---
**What a negative result here would mean.** Nothing in this project has shown a validated
directional edge across nine entry families, eight level types, 62 candlestick patterns, 22 chart
patterns, six timeframes and two instruments — and a trendline break is a close relative of the
level-break family already measured across 320 cells and found to carry none.

If stage 2 comes back at a 50% hit rate and a `vs_null20` that straddles zero, that is the expected
answer and it is worth having: it closes the question with evidence instead of leaving it open. The
settings from stage 1 remain useful for **seeing** structure regardless — that is what notebook 15 is
for.